# Creating DataFrames
We will create `DataFrame` objects from other data structures in Python, by reading in a CSV file, and by querying a database.

## About the Data
In this notebook, we will be working with earthquake data from September 18, 2018 - October 13, 2018 (obtained from the US Geological Survey (USGS) using the [USGS API](https://earthquake.usgs.gov/fdsnws/event/1/))

## Imports

**한글 요약**

- 데이터프레임(표)을 만드는 여러 방법을 보여주는 노트북
- 실전에서 제일 많이 쓰는 건 딱 하나: `pd.read_csv('파일')` → csv 파일을 표로 읽어오기
- 나머지(딕셔너리, 리스트, 튜플, numpy로 만들기)는 "이렇게도 만들 수 있다" 정도로만 보면 됨
- `!`로 시작하는 줄은 파이썬이 아니라 터미널 명령어. 파일을 읽기 전에 미리 살펴보는 용도라 몰라도 됨 (윈도우에서는 안 될 수도 있음)
- 마지막 DB(데이터베이스) 부분도 참고용

In [ ]:
# 필요한 도구들 불러오기
# datetime: 날짜/시간 다루는 도구 → dt로 줄여 부름
# numpy → np, pandas → pd (이건 거의 공식 같은 관례)
import datetime as dt
import numpy as np
import pandas as pd


## Creating a `Series` object

In [ ]:
# np.random.rand(5) → 0~1 사이 랜덤 소수 5개 만듦
# np.random.seed(0) → 랜덤의 '씨앗(seed)'을 0으로 고정. 이걸 하면 랜덤인데도 매번 똑같은 숫자가 나옴
#   (책이랑 내 결과가 같게 하려고. 안 하면 실행마다 다른 숫자 나옴)
np.random.seed(0) # set a seed for reproducibility
# 랜덤 숫자 5개로 'random'이라는 이름의 시리즈 만들기
pd.Series(np.random.rand(5), name='random')


0    0.548814
1    0.715189
2    0.602763
3    0.544883
4    0.423655
Name: random, dtype: float64

## Creating a `DataFrame` object from a `Series` object
Use the `to_frame()` method:

In [ ]:
# np.linspace(0, 10, num=5) → 0~10을 똑같은 간격으로 5개: 0, 2.5, 5, 7.5, 10
# pd.Series(...) → 그걸 시리즈로 만듦
# .to_frame() → 시리즈(열 1개)를 데이터프레임(표)으로 바꿈
# 이름을 안 줬으니 열 이름이 0으로 나옴
pd.Series(np.linspace(0, 10, num=5)).to_frame()


,0
0,0.0
1,2.5
2,5.0
3,7.5
4,10.0


## Creating a `DataFrame` from Python Data Structures
### From a dictionary of list-like structures
The dictionary values can be lists, NumPy arrays, etc. as long as they have length (generators don't have length so we can't use them here):

In [ ]:
np.random.seed(0) # set seed so result is reproducible (랜덤 고정)
# 딕셔너리로 데이터프레임 만들기: {열이름: 값들}. 딕셔너리 키 하나 = 열 하나
# 'random' 열 → 랜덤 소수 5개
# 'text' 열 → 문자열 리스트. 마지막 None은 '값 없음'이라는 뜻 (표에서 None 또는 NaN으로 보임)
# 'truth' 열 → True/False 중 랜덤으로 5개 뽑음
#   np.random.choice([True, False]) → 둘 중 하나 랜덤 선택
#   for _ in range(5) → 5번 반복. _ 는 "변수 이름은 필요 없어, 그냥 반복만 할게"라는 관례
# index=... → 행 이름표를 직접 정해줌. 여기서는 날짜로!
#   pd.date_range(end=2019년 4월 21일, freq='1D', periods=5) → 4/21에 끝나는, 하루(1 Day) 간격, 5개 날짜
#   → 4/17, 4/18, 4/19, 4/20, 4/21
#   name='date' → 인덱스에 'date'라는 이름 붙임 (결과 표 왼쪽 위에 date라고 나옴)
pd.DataFrame(
    {
        'random': np.random.rand(5),
        'text': ['hot', 'warm', 'cool', 'cold', None],
        'truth': [np.random.choice([True, False]) for _ in range(5)]
    },
    index=pd.date_range(
        end=dt.date(2019, 4, 21),
        freq='1D',
        periods=5,
        name='date'
    )
)


,random,text,truth
date,,,
2019-04-17,0.548814,hot,False
2019-04-18,0.715189,warm,True
2019-04-19,0.602763,cool,True
2019-04-20,0.544883,cold,False
2019-04-21,0.423655,None,True


### From a list of dictionaries

In [ ]:
# 딕셔너리들이 든 리스트로 만들기. 딕셔너리 하나 = 행 하나
# 각 딕셔너리의 키(mag, place)가 열 이름이 되고, 값이 그 행의 값이 됨
# 위 방식은 열 단위로 넣은 거고, 이번엔 행 단위로 넣은 거임
pd.DataFrame([
    {'mag': 5.2, 'place': 'California'},
    {'mag': 1.2, 'place': 'Alaska'},
    {'mag': 0.2, 'place': 'California'},
])


,mag,place
0,5.2,California
1,1.2,Alaska
2,0.2,California


### From a list of tuples

In [ ]:
# 튜플들이 든 리스트 만들기 (튜플 = 수정 못하는 리스트, ()로 만듦)
# range(5) → 0,1,2,3,4
# n**2 → n의 2제곱, n**3 → n의 3제곱 (** 는 거듭제곱)
# n=2일 때 (2, 4, 8) 이런 식으로 튜플 5개가 만들어짐
list_of_tuples = [(n, n**2, n**3) for n in range(5)]
list_of_tuples


[(0, 0, 0), (1, 1, 1), (2, 4, 8), (3, 9, 27), (4, 16, 64)]

In [ ]:
# 튜플 리스트로 데이터프레임 만들기. 튜플 하나 = 행 하나
# 튜플에는 열 이름이 없으니까 columns=[...]로 열 이름을 직접 정해줌
pd.DataFrame(
    list_of_tuples,
    columns=['n', 'n_squared', 'n_cubed']
)


,n,n_squared,n_cubed
0,0,0,0
1,1,1,1
2,2,4,8
3,3,9,27
4,4,16,64


### From a NumPy array

In [ ]:
# numpy 2차원 배열(리스트 안에 리스트)로 만들기. 안쪽 리스트 하나 = 행 하나
# 결과는 바로 위랑 완전히 같음. 만드는 재료만 다른 거임
pd.DataFrame(
    np.array([
        [0, 0, 0],
        [1, 1, 1],
        [2, 4, 8],
        [3, 9, 27],
        [4, 16, 64]
    ]), columns=['n', 'n_squared', 'n_cubed']
)


,n,n_squared,n_cubed
0,0,0,0
1,1,1,1
2,2,4,8
3,3,9,27
4,4,16,64


## Creating a `DataFrame` object from the contents of a CSV File

### Finding information on the file before reading it in
Before attempting to read in a file, we can use the command line to see important information about the file that may determine how we read it in. We can run command line code from Jupyter Notebooks (thanks to IPython) by using `!` before the code.

#### Number of lines (row count)
For example, we can find out how many lines are in the file by using the `wc` utility (word count) and counting lines in the file (`-l`). The file has 9,333 lines:

In [ ]:
# ! 로 시작하면 파이썬이 아니라 '터미널(명령 프롬프트) 명령어'를 실행하는 거임 (주피터/코랩 기능)
# wc = word count(단어 세기), -l = line(줄) 수만 세줘
# → 파일이 9333줄이라는 뜻 (첫 줄은 열 이름이니까 실제 데이터는 9332행)
# ※ 윈도우에서는 안 될 수 있음. 아래 마크다운 설명에 윈도우용 코드 있음
!wc -l data/earthquakes.csv


9333 data/earthquakes.csv


**Windows users**: if the above doesn't work for you (depends on your setup), then use this instead:

```python
!find /c /v "" data\earthquakes.csv
```



#### File size
We can find the file size by using `ls` to list the files in the `data` directory, and passing in the flags `-lh` to include the file size in human readable format. Then we use `grep` to find the file in question. Note that `|` passes the result of `ls` to `grep`. The `grep` utility is used for finding items that match patterns.

This tells us the file is 3.4 MB:

In [ ]:
# ls = list(폴더 안 파일 목록 보여줘), -lh = 자세히(l) + 사람이 읽기 쉬운 크기 단위로(h)
# | (파이프) = 왼쪽 결과를 오른쪽 명령어로 넘겨줘
# grep earthquakes.csv = 그 중에서 earthquakes.csv가 들어간 줄만 찾아줘
# → 3.4M = 파일 크기가 3.4MB라는 뜻
!ls -lh data | grep earthquakes.csv


-rw-r--r-- 1 stefanie stefanie 3.4M Aug 24 17:39 earthquakes.csv


**Windows users**: if the above doesn't work for you (depends on your setup), then use this instead:

```python
!dir data | findstr "earthquakes.csv"
```

We can even capture the result of a command and use it in our Python code:

In [ ]:
# 터미널 명령어 결과를 파이썬 변수에 저장할 수도 있음
# files = 폴더 안 파일 목록 (한 줄이 리스트 요소 하나)
files = !ls -lh data
# 그 중 'earthquake'라는 글자가 들어간 것만 골라냄 (리스트 컴프리헨션 + if 조건)
[file for file in files if 'earthquake' in file]


['-rw-r--r-- 1 stefanie stefanie 3.4M Aug 24 17:39 earthquakes.csv']

**Windows users**: if the above doesn't work for you (depends on your setup), then use this instead:

```python
files = !dir data
[file for file in files if 'earthquake' in file]
```


#### Examining a few rows
We can use `head` to look at the top `n` rows of the file. With the `-n` flag, we can specify how many. This shows use that the first row of the file contains headers and that it is comma-separated (just because the file extension is `.csv` doesn't it contains comma-separated values):

In [ ]:
# head = 파일의 위쪽 몇 줄만 보여줘, -n 2 = 2줄만
# 첫 줄 = 열 이름들(alert, cdi, code, ...), 둘째 줄 = 첫 번째 데이터
# 이걸 보면 값들이 ,(콤마)로 구분되어 있다는 걸 알 수 있음
!head -n 2 data/earthquakes.csv


alert,cdi,code,detail,dmin,felt,gap,ids,mag,magType,mmi,net,nst,place,rms,sig,sources,status,time,title,tsunami,type,types,tz,updated,url
,,37389218,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=ci37389218&format=geojson,0.008693,,85.0,",ci37389218,",1.35,ml,,ci,26.0,"9km NE of Aguanga, CA",0.19,28,",ci,",automatic,1539475168010,"M 1.4 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.0,1539475395144,https://earthquake.usgs.gov/earthquakes/eventpage/ci37389218


**Windows users**: if the above doesn't work for you (depends on your setup), then use this instead:

```python
n = 2
with open('data/earthquakes.csv', 'r') as file:
    for _ in range(n):
        print(file.readline(), end='\r')
```


Just like `head` gives rows from the top, `tail` gives rows from the bottom. This can help us check that there is no extraneous data on the bottom of the field, like perhaps some metadata about the fields that actually isn't part of the dataset:

In [ ]:
# tail = 파일의 아래쪽 몇 줄, -n 1 = 마지막 1줄
# 파일 끝에 이상한 게 붙어있지 않은지 확인용
!tail -n 1 data/earthquakes.csv


,,38063935,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=ci38063935&format=geojson,0.01698,,39.0,",ci38063935,",0.66,ml,,ci,24.0,"9km NE of Aguanga, CA",0.1,7,",ci,",reviewed,1537228864470,"M 0.7 - 9km NE of Aguanga, CA",0,earthquake,",focal-mechanism,geoserve,nearby-cities,origin,phase-data,scitech-link,",-480.0,1537305830770,https://earthquake.usgs.gov/earthquakes/eventpage/ci38063935


**Windows users**: if the above doesn't work for you (depends on your setup), then use this instead:

```python
import os

with open('data/earthquakes.csv', 'rb') as file:
    file.seek(0, os.SEEK_END)
    while file.read(1) != b'\n':
        file.seek(-2, os.SEEK_CUR)
    print(file.readline().decode())
```

*Note*: To inspect more than one row from the end of the file, you will have to use this instead, which requires reading the whole file:

```python
n = 2
with open('data/earthquakes.csv', 'r') as file:
    print('\r'.join(file.readlines()[-n:]))
```

#### Column count
We can use `awk` to find the column count. This is a utility for pattern scanning and processing. The `-F` flag allows us to specify the delimiter (comma, in this case). Then we specify what to do for each record in the file. We choose to print `NF` which is a predefined variable whose value is the number of fields in the current record. Here, we say `exit` so that we print the number of fields (columns, here) in the first row of the file, then we stop.

This tells us we have 26 data columns:

In [ ]:
# awk = 텍스트를 잘라서 처리하는 명령어 (좀 어려움, 이런 게 있다 정도만)
# -F',' = 콤마 기준으로 자르고
# '{print NF; exit}' = 잘린 칸 개수(NF = Number of Fields)를 출력하고 바로 종료
# → 첫 줄이 26칸 = 열이 26개
!awk -F',' '{print NF; exit}' data/earthquakes.csv


26


**Windows users**: if the above or below don't work for you (depends on your setup), then use this instead:

```python
with open('data/earthquakes.csv', 'r') as file:
    print(len(file.readline().split(',')))
```


Since we know the 1st line of the file had headers, and the file is comma-separated, we can also count the columns by using `head` to get headers and parsing them in Python:

In [ ]:
# 파이썬으로 열 개수 세기
# headers = 첫 줄(열 이름 줄)을 가져옴. 리스트 형태라 headers[0]이 실제 문자열
headers = !head -n 1 data/earthquakes.csv
# .split(',') → 콤마 기준으로 잘라서 리스트로 만듦
# len(...) → 그 리스트 길이 = 열 개수 = 26
len(headers[0].split(','))


26

**Windows users**: if you had to use the alternatives above, consider trying out [Cygwin](https://www.cygwin.com) or [Windows Subsystem for Linux (WSL)](https://docs.microsoft.com/en-us/windows/wsl/about).

### Reading in the file
Our file is small in size, has headers in the first row, and is comma-separated, so we don't need to provide any additional arguments to read in the file with `pd.read_csv()`, but be sure to check the [documentation](http://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html) for possible arguments:

In [ ]:
# pd.read_csv('파일 위치') → csv 파일을 읽어서 데이터프레임으로 만듦. 제일 많이 쓰는 명령어!
# 첫 줄을 자동으로 열 이름으로 인식하고, 콤마로 잘라줌
# 결과를 df라는 변수에 저장 (출력은 안 됨. 저장만)
df = pd.read_csv('data/earthquakes.csv')


Note that we can also pass in a URL. Let's read this same file from GitHub:

In [ ]:
# 파일 위치 대신 인터넷 주소(URL)를 넣어도 읽어옴
# 문자열 3개를 나눠 썼는데, 파이썬은 따옴표 문자열을 나란히 쓰면 자동으로 이어붙여줌
# ('https://github.com/stefmolin/' + 'Hands-On-...' + '/blob/...' 이렇게 한 줄로 된 거임. 그냥 길어서 나눠 쓴 것)
df = pd.read_csv(
    'https://github.com/stefmolin/'
    'Hands-On-Data-Analysis-with-Pandas-2nd-edition'
    '/blob/master/ch_02/data/earthquakes.csv?raw=True'
)


Pandas is usually very good at figuring out which options to use based on the input data, so we often won't need to add arguments to the call; however, there are many options available should we need them, some of which include the following:

| Parameter | Purpose |
| --- | --- |
| `sep` | Specifies the delimiter |
| `header` | Row number where the column names are located; the default option has `pandas` infer whether they are present |
| `names` | List of column names to use as the header |
| `index_col` | Column to use as the index |
| `usecols` | Specifies which columns to read in |
| `dtype` | Specifies data types for the columns |
| `converters` | Specifies functions for converting data in certain columns |
| `skiprows` | Rows to skip |
| `nrows` | Number of rows to read at a time (combine with `skiprows` to read a file bit by bit) |
| `parse_dates` | Automatically parse columns containing dates into datetime objects |
| `chunksize` | For reading the file in chunks |
| `compression` | For reading in compressed files without extracting beforehand |
| `encoding` | Specifies the file encoding |

## Writing a `DataFrame` Object to a CSV File
Note that the index of `df` is just row numbers, so we don't want to keep it. Therefore, we pass `index=False` to the `to_csv()` method:

In [ ]:
# to_csv → 데이터프레임을 csv 파일로 저장 (read_csv의 반대)
# index=False → 왼쪽 인덱스(0,1,2,...)는 그냥 행 번호니까 파일에 저장 안 함
#   (이거 안 쓰면 다음에 읽을 때 'Unnamed: 0'이라는 쓸모없는 열이 생김. 거의 항상 index=False 씀)
df.to_csv('output.csv', index=False)


## Writing a `DataFrame` Object to a Database
Note the `if_exists` parameter. By default, it will give you an error if you try to write a table that already exists. Here, we don't care if it is overwritten. Lastly, if we are interested in appending new rows, we set that to `'append'`.

In [ ]:
# sqlite3 → 파이썬에 내장된 데이터베이스(DB) 도구. 파일 하나로 되는 간단한 DB
import sqlite3

# with ... as connection: → DB에 연결하고, 블록(들여쓰기 부분)이 끝나면 자동으로 연결을 닫아줌
# sqlite3.connect('data/quakes.db') → quakes.db 파일에 연결 (없으면 새로 만듦)
with sqlite3.connect('data/quakes.db') as connection:
    # tsunamis.csv를 읽어서 → DB 안에 'tsunamis'라는 테이블(표)로 저장
    # to_sql('테이블이름', 연결, ...)
    # index=False → 인덱스는 저장 안 함
    # if_exists='replace' → 이미 같은 이름 테이블이 있으면 덮어써 (기본값은 에러 냄, 'append'면 뒤에 추가)
    pd.read_csv('data/tsunamis.csv').to_sql(
        'tsunamis', connection, index=False, if_exists='replace'
    )


## Creating a `DataFrame` Object by Querying a Database
Using a SQLite database. Otherwise you need to install [SQLAlchemy](https://www.sqlalchemy.org/).

In [ ]:
import sqlite3

with sqlite3.connect('data/quakes.db') as connection:
    # pd.read_sql('SQL 문장', 연결) → DB에 질문(쿼리)해서 결과를 데이터프레임으로 받음
    # 'SELECT * FROM tsunamis' → tsunamis 테이블에서 전부(*) 가져와 (SQL이라는 DB용 언어)
    tsunamis = pd.read_sql('SELECT * FROM tsunamis', connection)

# .head() → 위에서 5행만 보여줘 (표가 크면 다 보기 힘드니까). 앞으로 진짜 많이 쓸 거임
tsunamis.head()


,alert,type,title,place,magType,mag,time
0,None,earthquake,"M 5.0 - 165km NNW of Flying Fish Cove, Christm...","165km NNW of Flying Fish Cove, Christmas Island",mww,5.0,1539459504090
1,green,earthquake,"M 6.7 - 262km NW of Ozernovskiy, Russia","262km NW of Ozernovskiy, Russia",mww,6.7,1539429023560
2,green,earthquake,"M 5.6 - 128km SE of Kimbe, Papua New Guinea","128km SE of Kimbe, Papua New Guinea",mww,5.6,1539312723620
3,green,earthquake,"M 6.5 - 148km S of Severo-Kuril'sk, Russia","148km S of Severo-Kuril'sk, Russia",mww,6.5,1539213362130
4,green,earthquake,"M 6.2 - 94km SW of Kokopo, Papua New Guinea","94km SW of Kokopo, Papua New Guinea",mww,6.2,1539208835130


<hr>
<div>
    <a href="./1-pandas_data_structures.ipynb">
        <button style="float: left;">&#8592; Previous Notebook</button>
    </a>
    <a href="./3-making_dataframes_from_api_requests.ipynb">
        <button style="float: right;">Next Notebook &#8594;</button>
    </a>
</div>
<br>
<hr>